In [ ]:
import pandas as pd

df_raw = pd.read_csv('Big5_playerdata.csv')
df_raw["Total_Mis"] = df_raw["Total_Att"] - df_raw["Total_Cmp"]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from scipy import stats

df_esp = df_raw[df_raw.Comp == 'es La Liga']

possession_ending_events = [
    'Standard_Sh',           # Lövések
    'Take-Ons_Att',          # Dribling kísérletek
    'Take-Ons_Tkld',         # Sikertelen driblingek
    'Carries_Mis',           # Elvesztett labdák
    'Passes_Att',            # Passz kísérletek (csak mezőnyjátékosoknak)
    'Total_Att',             # Összes passz kísérlet
    'Outcomes_Off',          # Lesek
    'Outcomes_Blocks',       # Blokkolt passzok/lövések
    'Challenges_Lost',       # Elvesztett párharcok
    'Err'                    # Hibák
]

def calculate_usage_rate(df):
    """
    Calculate Usage Rate for each player
    """
    # Possession ending events meghatározása
    df['possession_ending_events'] = (
        df['Standard_Sh'].fillna(0) +
        df['Take-Ons_Att'].fillna(0) + 
        df['Take-Ons_Tkld'].fillna(0) +
        df['Carries_Mis'].fillna(0) +
        df['Total_Att'].fillna(0) * 0.3 +  # Passzok egy része possession ending
        df['Outcomes_Off'].fillna(0) +
        df['Outcomes_Blocks'].fillna(0) +
        df['Challenges_Lost'].fillna(0) +
        df['Err'].fillna(0)
    )
    
    # Csapatonkénti összes possession számolása
    team_possessions = df.groupby('Squad')['possession_ending_events'].sum().reset_index()
    team_possessions.columns = ['Squad', 'team_total_possessions']
    
    # Merge team possessions back to main df
    df = df.merge(team_possessions, on='Squad', how='left')
    
    # Usage rate számolása
    df['usage_rate'] = df['possession_ending_events'] / df['team_total_possessions']
    
    # Per 90 minutes adjustment
    df['usage_rate_per90'] = df['usage_rate'] * (90 / df['Playing Time_Min'])
    
    return df

def analyze_efficiency(df):
    """
    Analyze efficiency by comparing usage rate to output
    """
    # Output metrikák - választható: G+A, SCA, xG+xAG
    df['output_ga'] = df['Performance_Gls'].fillna(0) + df['Performance_Ast'].fillna(0)
    df['output_sca'] = df['SCA_SCA'].fillna(0)
    df['output_xgxag'] = df['Expected_xG'].fillna(0) + df['Expected_xAG'].fillna(0)
    
    # Per 90 rates
    df['output_ga_per90'] = df['output_ga'] / (df['Playing Time_Min'] / 90)
    df['output_sca_per90'] = df['output_sca'] / (df['Playing Time_Min'] / 90)
    df['output_xgxag_per90'] = df['output_xgxag'] / (df['Playing Time_Min'] / 90)
    
    return df

def plot_usage_vs_output(df, output_metric='output_ga_per90', min_minutes=450):
    """
    Plot usage rate vs output with worth-it line
    """
    # Filter players with sufficient minutes
    df_filtered = df[df['Playing Time_Min'] >= min_minutes].copy()
    
    # Calculate worth-it line (median efficiency)
    median_usage = df_filtered['usage_rate_per90'].median()
    median_output = df_filtered[output_metric].median()
    
    # Create scatter plot
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(df_filtered['usage_rate_per90'], 
                         df_filtered[output_metric],
                         alpha=0.6, s=50)
    
    # Add worth-it lines
    plt.axhline(y=median_output, color='red', linestyle='--', alpha=0.7, 
               label=f'Átlagos output: {median_output:.2f}')
    plt.axvline(x=median_usage, color='blue', linestyle='--', alpha=0.7,
               label=f'Átlagos usage: {median_usage:.4f}')
    
    # Quadrant labels
    plt.text(median_usage * 1.1, median_output * 1.1, 'Hatékony', fontsize=12, 
            bbox=dict(facecolor='green', alpha=0.2))
    plt.text(median_usage * 0.6, median_output * 0.6, 'Keveset használ', fontsize=12,
            bbox=dict(facecolor='blue', alpha=0.2))
    plt.text(median_usage * 1.1, median_output * 0.6, 'Pazarló', fontsize=12,
            bbox=dict(facecolor='red', alpha=0.2))
    plt.text(median_usage * 0.6, median_output * 1.1, 'Hatékony + kreatív', fontsize=12,
            bbox=dict(facecolor='purple', alpha=0.2))
    
    plt.xlabel('Usage Rate per 90')
    plt.ylabel('G+A per 90' if output_metric == 'output_ga_per90' else 
              'SCA per 90' if output_metric == 'output_sca_per90' else 'xG+xAG per 90')
    plt.title('Usage Rate vs Output Efficiency')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Add player labels for top performers
    top_players = df_filtered.nlargest(10, output_metric)
    for i, row in top_players.iterrows():
        plt.annotate(row['Player'], 
                    (row['usage_rate_per90'], row[output_metric]),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    plt.show()

def contextual_analysis(df):
    """
    Perform contextual analysis considering team possession and player roles
    """
    # Team possession context
    team_possession_stats = df.groupby('Squad').agg({
        'usage_rate_per90': 'mean',
        'Touches_Att 3rd': 'mean',
        'Touches_Att Pen': 'mean'
    }).reset_index()
    
    team_possession_stats.columns = ['Squad', 'avg_team_usage', 'avg_att_3rd_touches', 'avg_att_pen_touches']
    
    # Player role analysis
    df['position_group'] = df['Pos'].str.split('-').str[0]  # Első pozíció
    position_stats = df.groupby('position_group').agg({
        'usage_rate_per90': 'median',
        'output_ga_per90': 'median',
        'Expected_xG': 'median',
        'Expected_xAG': 'median'
    }).reset_index()
    
    return team_possession_stats, position_stats

# Fő elemzési folyamat
def main_analysis(df_raw):
    # 1. Usage rate számolás
    df = calculate_usage_rate(df_raw)
    
    # 2. Output metrikák hozzáadása
    df = analyze_efficiency(df)
    
    # 3. Plotok készítése
    print("Usage Rate vs G+A per 90:")
    plot_usage_vs_output(df, 'output_ga_per90')
    
    print("\nUsage Rate vs SCA per 90:")
    plot_usage_vs_output(df, 'output_sca_per90')
    
    print("\nUsage Rate vs xG+xAG per 90:")
    plot_usage_vs_output(df, 'output_xgxag_per90')
    
    # 4. Kontextuális elemzés
    team_stats, position_stats = contextual_analysis(df)
    
    print("\nCsapat statisztikák:")
    print(team_stats.sort_values('avg_team_usage', ascending=False).head(10))
    
    print("\nPozíciónkénti statisztikák:")
    print(position_stats)
    
    return df

# Elemzés futtatása
df_analyzed = main_analysis(df_esp)

# További részletes elemzés lehetőségei
def advanced_analysis(df):
    """
    Speciális elemzések különböző pozíciókra és csapatokra
    """
    # Pozíciónkénti elemzés
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    positions_to_analyze = ['FW', 'MF', 'DF', 'GK'][:3]  # GK kihagyása
    
    for i, pos in enumerate(positions_to_analyze):
        pos_data = df[df['Pos'].str.contains(pos, na=False)]
        
        if len(pos_data) > 0:
            ax = axes[i//2, i%2]
            scatter = ax.scatter(pos_data['usage_rate_per90'], 
                               pos_data['output_ga_per90'],
                               alpha=0.6)
            
            ax.set_xlabel('Usage Rate per 90')
            ax.set_ylabel('G+A per 90')
            ax.set_title(f'Usage vs Output - {pos} pozíció')
            ax.grid(True, alpha=0.3)
            
            # Annotate top performers
            top_pos_players = pos_data.nlargest(5, 'output_ga_per90')
            for _, row in top_pos_players.iterrows():
                ax.annotate(row['Player'], 
                          (row['usage_rate_per90'], row['output_ga_per90']),
                          xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    plt.tight_layout()
    plt.show()

# Speciális elemzés futtatása
advanced_analysis(df_analyzed)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from scipy import stats

# Stílusbeállítások
plt.style.use('default')
sns.set_palette("viridis")

# Adatok betöltése (feltételezve, hogy df_raw már létezik)
# df_raw = pd.read_csv('your_data.csv')

# Possession ending events kiválasztása a megadott oszlopok közül
possession_ending_events = [
    'Standard_Sh',           # Lövések
    'Take-Ons_Att',          # Dribling kísérletek
    'Carries_Mis',           # Labda elvesztése cipelés közben
    'Pass Types_TB',         # Átütések (ami kockázatosabb passzok)
    'Outcomes_Off',          # Lesek
    'Outcomes_Blocks',       # Blokkolt passzok/lövések
    'Challenges_Lost',       # Párharcok elvesztése
    'Performance_Off',       # Szabálytalanságok
    'Performance_Lost'       # Labda elvesztése (ha van ilyen oszlop)
]

# Elérhető oszlopok szűrése
available_events = [col for col in possession_ending_events if col in df_raw.columns]

print("Possession ending events használva:")
print(available_events)

# Alap DataFrame létrehozása a szükséges oszlopokkal
df = df_esp[['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 
             'Playing Time_Min', '90s', 'Performance_Gls', 
             'Performance_Ast', 'Performance_G+A'] + available_events].copy()

# Possession ending events összegzése
df['Possession_Ending_Events'] = df[available_events].sum(axis=1)

# Csapat szintű adatok összesítése a possession számoláshoz
team_possession = df.groupby('Squad')['Possession_Ending_Events'].sum().reset_index()
team_possession.columns = ['Squad', 'Team_Possession_Events']

# Csapat adatok mergelése
df = pd.merge(df, team_possession, on='Squad', how='left')

# Usage Rate számítás
df['Usage_Rate'] = (df['Possession_Ending_Events'] / df['Team_Possession_Events']) * 100
df['Usage_Rate_per_90'] = (df['Possession_Ending_Events'] / df['90s']) / (df['Team_Possession_Events'] / df.groupby('Squad')['90s'].transform('sum'))

# Output metrikák per 90
df['G+A_per_90'] = df['Performance_G+A'] / df['90s']
df['Goals_per_90'] = df['Performance_Gls'] / df['90s']
df['Assists_per_90'] = df['Performance_Ast'] / df['90s']

# Pozíciók csoportosítása
def categorize_position(pos):
    if isinstance(pos, str):
        if any(x in pos for x in ['FW', 'ST', 'CF']):
            return 'Forward'
        elif any(x in pos for x in ['MF', 'AM', 'CM', 'DM']):
            return 'Midfielder'
        elif any(x in pos for x in ['DF', 'CB', 'FB', 'WB']):
            return 'Defender'
        elif 'GK' in pos:
            return 'Goalkeeper'
    return 'Other'

df['Position_Category'] = df['Pos'].apply(categorize_position)

# Effektivitás metrika
df['Efficiency_Ratio'] = df['G+A_per_90'] / df['Usage_Rate_per_90'].replace(0, np.nan)

# Szűrés releváns pozíciókra és percekre
df_analysis = df[(df['Playing Time_Min'] >= 450) & 
                (df['Position_Category'].isin(['Forward', 'Midfielder']))].copy()

# Worth-it line számítás (median alapján)
median_usage = df_analysis['Usage_Rate_per_90'].median()
median_output = df_analysis['G+A_per_90'].median()

# Vizualizációk
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

# 1. Usage Rate vs G+A per 90
scatter = ax1.scatter(df_analysis['Usage_Rate_per_90'], df_analysis['G+A_per_90'],
                     c=df_analysis['Efficiency_Ratio'], cmap='viridis', alpha=0.7, s=50)
ax1.axhline(y=median_output, color='red', linestyle='--', alpha=0.7, label='Median Output')
ax1.axvline(x=median_usage, color='blue', linestyle='--', alpha=0.7, label='Median Usage')
ax1.set_xlabel('Usage Rate per 90')
ax1.set_ylabel('Goals + Assists per 90')
ax1.set_title('Usage Rate vs Output Efficiency')
ax1.legend()
plt.colorbar(scatter, ax=ax1, label='Efficiency Ratio')

# 2. Pozíciónkénti eloszlás
positions = df_analysis['Position_Category'].unique()
colors = plt.cm.viridis(np.linspace(0, 1, len(positions)))

for i, pos in enumerate(positions):
    subset = df_analysis[df_analysis['Position_Category'] == pos]
    ax2.scatter(subset['Usage_Rate_per_90'], subset['G+A_per_90'],
               color=colors[i], label=pos, alpha=0.7)

ax2.set_xlabel('Usage Rate per 90')
ax2.set_ylabel('Goals + Assists per 90')
ax2.set_title('Usage by Position')
ax2.legend()

# 3. Efficiency Ratio eloszlás
efficiency_data = df_analysis[df_analysis['Efficiency_Ratio'] < df_analysis['Efficiency_Ratio'].quantile(0.95)]
ax3.hist(efficiency_data['Efficiency_Ratio'], bins=30, alpha=0.7, edgecolor='black')
ax3.axvline(efficiency_data['Efficiency_Ratio'].median(), color='red', linestyle='--', label='Median Efficiency')
ax3.set_xlabel('Efficiency Ratio (G+A per Usage)')
ax3.set_ylabel('Frequency')
ax3.set_title('Efficiency Ratio Distribution')
ax3.legend()

# 4. Legjobb és legrosszabb teljesítők
top_10 = df_analysis.nlargest(10, 'Efficiency_Ratio')
bottom_10 = df_analysis.nsmallest(10, 'Efficiency_Ratio')

ax4.barh(range(10), top_10['Efficiency_Ratio'], alpha=0.7, label='Most Efficient')
ax4.barh(range(10, 20), bottom_10['Efficiency_Ratio'], alpha=0.7, label='Least Efficient')
ax4.set_yticks(range(20))
ax4.set_yticklabels(list(top_10['Player']) + list(bottom_10['Player']))
ax4.set_xlabel('Efficiency Ratio')
ax4.set_title('Top 10 Most and Least Efficient Players')
ax4.legend()

plt.tight_layout()
plt.show()

# Részletes elemzés
print("="*50)
print("ÁTLAGOS METRIKÁK POZÍCIÓNKÉNT:")
print("="*50)
position_stats = df_analysis.groupby('Position_Category').agg({
    'Usage_Rate_per_90': 'mean',
    'G+A_per_90': 'mean',
    'Efficiency_Ratio': 'mean',
    'Playing Time_Min': 'count'
}).round(3)
print(position_stats)

print("\n" + "="*50)
print("LEGHATÉKONYABB JÁTÉKOSOK:")
print("="*50)
most_efficient = df_analysis.nlargest(10, 'Efficiency_Ratio')[['Player', 'Squad', 'Pos', 
                                                              'Usage_Rate_per_90', 
                                                              'G+A_per_90', 
                                                              'Efficiency_Ratio']]
print(most_efficient.round(3))

print("\n" + "="*50)
print("LEGHATÉKONYABB CSAPATOK ÁTLAGOS HASZNÁLATI ARÁNYA:")
print("="*50)
team_efficiency = df_analysis.groupby('Squad').agg({
    'Usage_Rate_per_90': 'mean',
    'G+A_per_90': 'mean',
    'Efficiency_Ratio': 'mean',
    'Player': 'count'
}).round(3).sort_values('Efficiency_Ratio', ascending=False)
print(team_efficiency.head(10))

# További elemzés: korrelációk
correlation_matrix = df_analysis[['Usage_Rate_per_90', 'G+A_per_90', 'Goals_per_90', 
                                 'Assists_per_90', 'Efficiency_Ratio']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Metrika Korrelációk')
plt.show()

# Playstyle klaszterezés
from sklearn.cluster import KMeans

# Adatok előkészítése klaszterezéshez
X = df_analysis[['Usage_Rate_per_90', 'G+A_per_90']].dropna()
X_scaled = StandardScaler().fit_transform(X)

# K-means klaszterezés
kmeans = KMeans(n_clusters=4, random_state=42)
df_analysis.loc[X.index, 'Playstyle_Cluster'] = kmeans.fit_predict(X_scaled)

# Playstyle klaszterek értelmezése
playstyle_labels = {
    0: 'Low Usage, Low Output',
    1: 'High Usage, High Output',
    2: 'Low Usage, High Output (Efficient)',
    3: 'High Usage, Low Output (Inefficient)'
}

df_analysis['Playstyle'] = df_analysis['Playstyle_Cluster'].map(playstyle_labels)

# Playstyle eloszlás
plt.figure(figsize=(12, 8))
for style in df_analysis['Playstyle'].unique():
    subset = df_analysis[df_analysis['Playstyle'] == style]
    plt.scatter(subset['Usage_Rate_per_90'], subset['G+A_per_90'], 
               label=style, alpha=0.7, s=50)

plt.xlabel('Usage Rate per 90')
plt.ylabel('Goals + Assists per 90')
plt.title('Player Playstyles based on Usage and Output')
plt.legend()
plt.show()

print("="*50)
print("PLAYSTYLE ELOSZLÁS:")
print("="*50)
print(df_analysis['Playstyle'].value_counts())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Stílusbeállítások
plt.style.use('default')
sns.set_palette("viridis")

# Először ellenőrizzük és javítsuk az adatokat
print("DataFrame oszlopai és ismétlődések ellenőrzése...")
print(f"Oszlopok száma: {len(df_raw.columns)}")
print(f"Duplikált oszlopnevek: {df_raw.columns[df_raw.columns.duplicated()].tolist()}")

# Ha vannak duplikált oszlopnevek, javítsuk őket
if df_raw.columns.duplicated().any():
    print("Duplikált oszlopnevek találhatóak, javítás...")
    df_raw = df_raw.loc[:, ~df_raw.columns.duplicated()]

# Ellenőrizzük az indexet is
print(f"Duplikált indexek: {df_raw.index.duplicated().sum()}")
if df_raw.index.duplicated().any():
    print("Duplikált indexek találhatóak, resetelés...")
    df_raw = df_raw.reset_index(drop=True)

# Possession ending events kiválasztása - csak az elérhető oszlopok
possession_ending_events = [
    'Standard_Sh',           # Lövések
    'Take-Ons_Tkld',         # Dribling sikertelen
    'Carries_Mis',           # Labda elvesztése cipelés közben
    'Total_Mis',             # Sikertelen passzok
    'Outcomes_Off',          # Lesek
    'Challenges_Lost',       # Párharcok elvesztése
    'Performance_Off',       # Szabálytalanságok
]

# Elérhető oszlopok szűrése
available_events = [col for col in possession_ending_events if col in df_raw.columns]

print("Possession ending events használva:")
print(available_events)

# Alap DataFrame létrehozása - biztosítsuk, hogy minden oszlop létezik
required_columns = ['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 
                   'Playing Time_Min', '90s', 'Performance_Gls', 
                   'Performance_Ast', 'Performance_G+A',
                   'SCA_SCA', 'GCA_GCA', 'Carries_Carries', 'Take-Ons_Succ']

# Csak azokat az oszlopokat válasszuk, amelyek valóban léteznek
existing_columns = [col for col in required_columns + available_events if col in df_raw.columns]

df = df_esp[existing_columns].copy()

# Biztonsági másolat készítése és NaN értékek kezelése
df = df.copy()
df = df.fillna(0)

# Possession ending events összegzése - csak ha vannak elérhető események
if available_events:
    df['Possession_Ending_Events'] = df[available_events].sum(axis=1)
else:
    df['Possession_Ending_Events'] = df['Standard_Sh'] + df['Take-Ons_Att']  # alap események

# Csapat szintű adatok összesítése a possession számoláshoz
team_possession = df.groupby('Squad')['Possession_Ending_Events'].sum().reset_index()
team_possession.columns = ['Squad', 'Team_Possession_Events']

# Csapat adatok mergelése
df = pd.merge(df, team_possession, on='Squad', how='left')

# Usage Rate számítás - kerüljük el a 0-val való osztást
df['Usage_Rate_per_90'] = np.where(
    df['90s'] > 0, 
    df['Possession_Ending_Events'] / df['90s'], 
    0
)

# Output metrikák per 90 - biztonságos osztás
def safe_divide(a, b):
    return np.where(b > 0, a / b, 0)

df['G+A_per_90'] = safe_divide(df['Performance_G+A'], df['90s'])
df['Goals_per_90'] = safe_divide(df['Performance_Gls'], df['90s'])
df['Assists_per_90'] = safe_divide(df['Performance_Ast'], df['90s'])
df['SCA_per_90'] = safe_divide(df.get('SCA_SCA', 0), df['90s'])
df['GCA_per_90'] = safe_divide(df.get('GCA_GCA', 0), df['90s'])
df['Carries_per_90'] = safe_divide(df.get('Carries_Carries', 0), df['90s'])
df['Successful_TakeOns_per_90'] = safe_divide(df.get('Take-Ons_Succ', 0), df['90s'])

# Shots per 90 - csak ha az oszlop létezik
if 'Standard_Sh' in df.columns:
    df['Shots_per_90'] = safe_divide(df['Standard_Sh'], df['90s'])
else:
    df['Shots_per_90'] = 0

if 'Standard_SoT' in df.columns:
    df['Shots_on_Target_per_90'] = safe_divide(df['Standard_SoT'], df['90s'])
else:
    df['Shots_on_Target_per_90'] = 0

# Effektivitás metrikák - kerüljük el a 0-val való osztást
df['Efficiency_Ratio'] = np.where(
    df['Usage_Rate_per_90'] > 0,
    df['G+A_per_90'] / df['Usage_Rate_per_90'],
    0
)
df['SCA_Efficiency'] = np.where(
    df['Usage_Rate_per_90'] > 0,
    df['SCA_per_90'] / df['Usage_Rate_per_90'],
    0
)

# Pozíciók csoportosítása
def categorize_position(pos):
    if isinstance(pos, str):
        if any(x in pos for x in ['FW', 'ST', 'CF', 'SS']):
            return 'Forward'
        elif any(x in pos for x in ['MF', 'AM', 'CM', 'DM', 'LM', 'RM', 'WM']):
            return 'Midfielder'
        elif any(x in pos for x in ['DF', 'CB', 'FB', 'WB', 'LB', 'RB']):
            return 'Defender'
        elif 'GK' in pos:
            return 'Goalkeeper'
    return 'Other'

df['Position_Category'] = df['Pos'].apply(categorize_position)

# Szűrés releváns pozíciókra és percekre
df_analysis = df[(df['Playing Time_Min'] >= 450) & 
                (df['Position_Category'].isin(['Forward', 'Midfielder']))].copy()

# KOMPLEX PLAYSTYLE KLASZTEREZÉS
def classify_playstyle(row):
    try:
        if row['Position_Category'] == 'Forward':
            # Forward pozíciók részletes klaszterezése
            row_usage_q = (row['Usage_Rate_per_90'] > df_analysis['Usage_Rate_per_90'].quantile(0.67))
            row_sca_q = (row['SCA_per_90'] > df_analysis['SCA_per_90'].quantile(0.67))
            row_goals_q = (row['Goals_per_90'] > df_analysis['Goals_per_90'].quantile(0.67))
            
            if not row_usage_q and not row_sca_q and row_goals_q:
                return 'Finisher'
            elif row_usage_q and row_sca_q:
                return 'Second Striker'
            else:
                return 'Complete Forward'
                
        elif row['Position_Category'] == 'Midfielder':
            # Midfielder pozíciók klaszterezése
            sca_quantile = (row['SCA_per_90'] > df_analysis['SCA_per_90'].quantile(0.67))
            usage_quantile = (row['Usage_Rate_per_90'] > df_analysis['Usage_Rate_per_90'].quantile(0.67))
            
            if sca_quantile and not usage_quantile:
                return 'Creator'
            elif sca_quantile and usage_quantile:
                return 'Playmaker'
            else:
                return 'Carrier'
        
        return 'Other'
    except:
        return 'Other'

# Playstyle classification
df_analysis['Playstyle'] = df_analysis.apply(classify_playstyle, axis=1)

# LOW-USAGE, HIGH-OUTPUT játékosok azonosítása
df_analysis['Usage_Quantile'] = pd.qcut(df_analysis['Usage_Rate_per_90'], 4, labels=['Very Low', 'Low', 'High', 'Very High'])
df_analysis['Output_Quantile'] = pd.qcut(df_analysis['G+A_per_90'], 4, labels=['Very Low', 'Low', 'High', 'Very High'])
df_analysis['SCA_Quantile'] = pd.qcut(df_analysis['SCA_per_90'], 4, labels=['Very Low', 'Low', 'High', 'Very High'])

low_usage_high_output = df_analysis[
    (df_analysis['Usage_Quantile'].isin(['Very Low', 'Low'])) &
    (df_analysis['Output_Quantile'].isin(['High', 'Very High']))
]

# Regressziós egyenesek számítása
# 1. Usage Rate vs G+A
x_ga = df_analysis['Usage_Rate_per_90'].values
y_ga = df_analysis['G+A_per_90'].values
z_scores_ga = np.abs(stats.zscore(np.column_stack([x_ga, y_ga]), nan_policy='omit'))
filtered_idx_ga = np.all(z_scores_ga < 3, axis=1)
slope_ga, intercept_ga, _, _, _ = stats.linregress(x_ga[filtered_idx_ga], y_ga[filtered_idx_ga])
worth_it_intercept_ga = intercept_ga * 1.2

# 2. Usage Rate vs SCA
x_sca = df_analysis['Usage_Rate_per_90'].values
y_sca = df_analysis['SCA_per_90'].values
z_scores_sca = np.abs(stats.zscore(np.column_stack([x_sca, y_sca]), nan_policy='omit'))
filtered_idx_sca = np.all(z_scores_sca < 3, axis=1)
slope_sca, intercept_sca, _, _, _ = stats.linregress(x_sca[filtered_idx_sca], y_sca[filtered_idx_sca])
worth_it_intercept_sca = intercept_sca * 1.2

# Játékosok klaszterezése efficiency alapján
df_analysis['Above_Worth_It_Line_GA'] = df_analysis['G+A_per_90'] > (slope_ga * df_analysis['Usage_Rate_per_90'] + worth_it_intercept_ga)
df_analysis['Above_Worth_It_Line_SCA'] = df_analysis['SCA_per_90'] > (slope_sca * df_analysis['Usage_Rate_per_90'] + worth_it_intercept_sca)

# VIZUALIZÁCIÓK
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 16))

# 1. Usage Rate vs G+A per 90 with Enhanced Playstyles
playstyle_colors = {
    'Finisher': 'red', 'Second Striker': 'purple', 'Complete Forward': 'orange',
    'Creator': 'blue', 'Playmaker': 'green', 'Carrier': 'gray'
}

for playstyle, color in playstyle_colors.items():
    subset = df_analysis[df_analysis['Playstyle'] == playstyle]
    if len(subset) > 0:
        ax1.scatter(subset['Usage_Rate_per_90'], subset['G+A_per_90'],
                   color=color, label=playstyle, alpha=0.7, s=60)

# Regressziós egyenes és Worth-It Line
x_range = np.linspace(df_analysis['Usage_Rate_per_90'].min(), df_analysis['Usage_Rate_per_90'].max(), 100)
ax1.plot(x_range, slope_ga * x_range + intercept_ga, 'k--', alpha=0.8, label='Átlagos Trend (G+A)')
ax1.plot(x_range, slope_ga * x_range + worth_it_intercept_ga, 'r-', linewidth=2, label='Worth-It Line (G+A)')

ax1.set_xlabel('Usage Rate per 90', fontsize=12)
ax1.set_ylabel('Goals + Assists per 90', fontsize=12)
ax1.set_title('Usage Rate vs G+A by Enhanced Playstyle\n(Worth-It Line Analysis)', fontsize=14)
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# 2. Usage Rate vs Shot-Creating Actions per 90
for playstyle, color in playstyle_colors.items():
    subset = df_analysis[df_analysis['Playstyle'] == playstyle]
    if len(subset) > 0:
        ax2.scatter(subset['Usage_Rate_per_90'], subset['SCA_per_90'],
                   color=color, label=playstyle, alpha=0.7, s=60)

# Regressziós egyenes és Worth-It Line for SCA
ax2.plot(x_range, slope_sca * x_range + intercept_sca, 'k--', alpha=0.8, label='Átlagos Trend (SCA)')
ax2.plot(x_range, slope_sca * x_range + worth_it_intercept_sca, 'r-', linewidth=2, label='Worth-It Line (SCA)')

ax2.set_xlabel('Usage Rate per 90', fontsize=12)
ax2.set_ylabel('Shot-Creating Actions per 90', fontsize=12)
ax2.set_title('Usage Rate vs SCA by Playstyle\n(Worth-It Line Analysis)', fontsize=14)
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.grid(True, alpha=0.3)

# 3. Playstyle eloszlás efficiency szerint
playstyle_efficiency_ga = []
playstyle_efficiency_sca = []
playstyles_ordered = ['Finisher', 'Complete Forward', 'Second Striker', 'Creator', 'Playmaker', 'Carrier']

for playstyle in playstyles_ordered:
    subset = df_analysis[df_analysis['Playstyle'] == playstyle]
    if len(subset) > 0:
        playstyle_efficiency_ga.append(subset['Efficiency_Ratio'].median())
        playstyle_efficiency_sca.append(subset['SCA_Efficiency'].median())

x_pos = np.arange(len(playstyles_ordered))
width = 0.35

colors_ga = [playstyle_colors[p] for p in playstyles_ordered]
colors_sca = [playstyle_colors[p] for p in playstyles_ordered]

bars1 = ax3.bar(x_pos - width/2, playstyle_efficiency_ga, width, 
               label='G+A Efficiency', alpha=0.7)
bars2 = ax3.bar(x_pos + width/2, playstyle_efficiency_sca, width, 
               label='SCA Efficiency', alpha=0.7)

ax3.set_xlabel('Playstyle', fontsize=12)
ax3.set_ylabel('Efficiency Ratio', fontsize=12)
ax3.set_title('Efficiency Comparison by Playstyle', fontsize=14)
ax3.set_xticks(x_pos)
ax3.set_xticklabels(playstyles_ordered, rotation=45, ha='right')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# 4. Low-Usage, High-Output játékosok playstyle szerint
low_usage_playstyle_counts = low_usage_high_output['Playstyle'].value_counts()
colors_low_usage = [playstyle_colors[p] for p in low_usage_playstyle_counts.index]

wedges, texts, autotexts = ax4.pie(low_usage_playstyle_counts.values, 
                                  labels=low_usage_playstyle_counts.index,
                                  colors=colors_low_usage, 
                                  autopct='%1.1f%%', startangle=90)
ax4.set_title('Low-Usage, High-Output Players by Playstyle', fontsize=14)

plt.tight_layout()
plt.show()

# RÉSZLETES ELEMZÉS
print("\n" + "="*70)
print("ENHANCED PLAYSTYLE ELEMZÉS")
print("="*70)
playstyle_stats = df_analysis.groupby('Playstyle').agg({
    'Usage_Rate_per_90': ['median', 'count'],
    'G+A_per_90': 'median',
    'Goals_per_90': 'median',
    'Assists_per_90': 'median',
    'SCA_per_90': 'median',
    'Efficiency_Ratio': 'median',
    'SCA_Efficiency': 'median'
}).round(3)
print(playstyle_stats)

print("\n" + "="*70)
print("TOP 5 LOW-USAGE, HIGH-OUTPUT JÁTÉKOSOK PLAYSTYLE SZERINT:")
print("="*70)
for playstyle in ['Finisher', 'Creator']:
    subset = low_usage_high_output[low_usage_high_output['Playstyle'] == playstyle]
    if len(subset) > 0:
        top_players = subset.nlargest(5, 'Efficiency_Ratio')[['Player', 'Squad', 'Pos',
                                                            'Usage_Rate_per_90', 
                                                            'G+A_per_90', 
                                                            'SCA_per_90',
                                                            'Efficiency_Ratio']]
        print(f"\n{playstyle}:")
        print(top_players.round(3))

print("\n" + "="*70)
print("FORWARD PLAYSTYLE COMPARISON:")
print("="*70)
forward_styles = ['Finisher', 'Complete Forward', 'Second Striker']
forward_stats = df_analysis[df_analysis['Playstyle'].isin(forward_styles)].groupby('Playstyle').agg({
    'Usage_Rate_per_90': 'median',
    'Goals_per_90': 'median',
    'SCA_per_90': 'median',
    'Shots_per_90': 'median',
    'Shots_on_Target_per_90': 'median',
    'Efficiency_Ratio': 'median'
}).round(3)
print(forward_stats)

print("\n" + "="*70)
print("WORTH-IT LINE PERFORMANCE:")
print("="*70)
worth_it_stats = pd.DataFrame({
    'Above_Worth_It_G+A': df_analysis['Above_Worth_It_Line_GA'].value_counts(),
    'Above_Worth_It_SCA': df_analysis['Above_Worth_It_Line_SCA'].value_counts()
})
print(worth_it_stats)

# Speciális metrikák a különböző playstyle-okhoz
print("\n" + "="*70)
print("PLAYSTYLE SPECIFIKUS JELLEMZŐK:")
print("="*70)
playstyle_characteristics = df_analysis.groupby('Playstyle').agg({
    'Goals_per_90': lambda x: f"{x.quantile(0.25):.2f}-{x.quantile(0.75):.2f}",
    'Assists_per_90': lambda x: f"{x.quantile(0.25):.2f}-{x.quantile(0.75):.2f}",
    'SCA_per_90': lambda x: f"{x.quantile(0.25):.2f}-{x.quantile(0.75):.2f}",
    'Usage_Rate_per_90': lambda x: f"{x.quantile(0.25):.2f}-{x.quantile(0.75):.2f}"
})
print(playstyle_characteristics)

# Legjobb játékosok minden playstyle-ban
print("\n" + "="*70)
print("BEST PLAYERS BY PLAYSTYLE (Efficiency):")
print("="*70)
for playstyle in playstyles_ordered:
    subset = df_analysis[df_analysis['Playstyle'] == playstyle]
    if len(subset) > 0:
        best_players = subset.nlargest(3, 'Efficiency_Ratio')[['Player', 'Squad', 
                                                             'Usage_Rate_per_90', 
                                                             'G+A_per_90', 
                                                             'SCA_per_90',
                                                             'Efficiency_Ratio']]
        print(f"\n{playstyle}:")
        print(best_players.round(3))